In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from math import radians, cos, sin, asin, sqrt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Helper function for geographic distance
def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2 - lon1, lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 2 * asin(sqrt(a)) * 6371

In [2]:
# 1. Load Data
df = pd.read_csv('../data/puerto_rico_master_profile.csv')
df['municipio'] = df['municipio'].str.replace(' Municipio', '', case=False).str.strip()
df = df.sort_values(by=['municipio', 'year']).reset_index(drop=True)

# 2. Add Coordinates
municipio_coords = {'Adjuntas': (18.1627, -66.7222), 'Aguada': (18.3605, -67.1882), 'Aguadilla': (18.4275, -67.1541), 'Aguas Buenas': (18.2572, -66.1031), 'Aibonito': (18.1408, -66.2663), 'Añasco': (18.2827, -67.1399), 'Arecibo': (18.4724, -66.7157), 'Arroyo': (17.9658, -66.0614), 'Barceloneta': (18.4505, -66.5385), 'Barranquitas': (18.1861, -66.3061), 'Bayamón': (18.3808, -66.1557), 'Cabo Rojo': (18.0866, -67.1457), 'Caguas': (18.2325, -66.0391), 'Camuy': (18.4838, -66.8450), 'Canóvanas': (18.3791, -65.9011), 'Carolina': (18.3808, -65.9574), 'Cataño': (18.4388, -66.1182), 'Cayey': (18.1119, -66.1663), 'Ceiba': (18.2633, -65.6474), 'Ciales': (18.3361, -66.4688), 'Cidra': (18.1758, -66.1617), 'Coamo': (18.0855, -66.3579), 'Comerío': (18.2183, -66.2257), 'Corozal': (18.3411, -66.3161), 'Culebra': (18.3030, -65.3010), 'Dorado': (18.4588, -66.2677), 'Fajardo': (18.3258, -65.6524), 'Florida': (18.3625, -66.5614), 'Guánica': (17.9716, -66.9080), 'Guayama': (17.9741, -66.1100), 'Guayanilla': (18.0130, -66.7919), 'Guaynabo': (18.3580, -66.1111), 'Gurabo': (18.2544, -65.9728), 'Hatillo': (18.4863, -66.8250), 'Hormigueros': (18.1397, -67.1274), 'Humacao': (18.1497, -65.8274), 'Isabela': (18.5011, -67.0247), 'Jayuya': (18.2186, -66.5916), 'Juana Díaz': (18.0525, -66.5064), 'Juncos': (18.2275, -65.9211), 'Lajas': (18.0497, -67.0594), 'Lares': (18.2947, -66.8771), 'Las Marías': (18.2514, -66.9922), 'Las Piedras': (18.1827, -65.8661), 'Loíza': (18.4330, -65.8797), 'Luquillo': (18.3725, -65.7166), 'Manatí': (18.4274, -66.4922), 'Maricao': (18.1808, -66.9799), 'Maunabo': (18.0072, -65.8994), 'Mayagüez': (18.2011, -67.1397), 'Moca': (18.3947, -67.1131), 'Morovis': (18.3258, -66.4053), 'Naguabo': (18.2119, -65.7350), 'Naranjito': (18.3008, -66.2447), 'Orocovis': (18.2269, -66.4411), 'Patillas': (18.0066, -66.0161), 'Peñuelas': (18.0622, -66.7219), 'Ponce': (18.0111, -66.6141), 'Quebradillas': (18.4738, -66.9385), 'Rincón': (18.3403, -67.2500), 'Río Grande': (18.3794, -65.8311), 'Sabana Grande': (18.0777, -66.9605), 'Salinas': (17.9775, -66.2974), 'San Germán': (18.0816, -67.0400), 'San Juan': (18.4655, -66.1057), 'San Lorenzo': (18.1894, -65.9611), 'San Sebastián': (18.3372, -66.9905), 'Santa Isabel': (17.9666, -66.4050), 'Toa Alta': (18.3883, -66.2485), 'Toa Baja': (18.4438, -66.2591), 'Trujillo Alto': (18.3547, -66.0072), 'Utuado': (18.2655, -66.7005), 'Vega Alta': (18.4119, -66.3314), 'Vega Baja': (18.4438, -66.3874), 'Vieques': (18.1261, -65.4400), 'Villalba': (18.1275, -66.4925), 'Yabucoa': (18.0505, -65.8794), 'Yauco': (18.0350, -66.8499)}
df['lat'] = df['municipio'].map(lambda x: municipio_coords.get(x, (None, None))[0])
df['lon'] = df['municipio'].map(lambda x: municipio_coords.get(x, (None, None))[1])

# 3. Process Shocks
hurricane_tracks = pd.read_csv('../data/natural_disasters/caribbean_hurricane_tracks_2010_2025.csv')
with open('../data/natural_disasters/puerto_rico_earthquakes.json') as f:
    eq_data = pd.DataFrame(json.load(f))
eq_data['year'] = pd.to_datetime(eq_data['date']).dt.year

def get_distances_vectorized(target_lat, target_lon, lats, lons):
    target_lat, target_lon = np.radians(target_lat), np.radians(target_lon)
    lats, lons = np.radians(lats), np.radians(lons)
    return 2 * np.arcsin(np.sqrt(np.sin((lats - target_lat)/2)**2 + np.cos(target_lat) * np.cos(lats) * np.sin((lons - target_lon)/2)**2)) * 6371

def get_shocks(row):
    if pd.isna(row['lat']): return 0, 0
    # Hurricane
    h_year = hurricane_tracks[hurricane_tracks['Year'] == row['year']]
    h_max = h_year.iloc[get_distances_vectorized(row['lat'], row['lon'], h_year['Lat'].values, h_year['Lon'].values) <= 100]['Wind_Knots'].max() if not h_year.empty else 0
    # Seismic
    s_year = eq_data[(eq_data['year'] == row['year']) & (eq_data['mag'] >= 4.0)]
    s_max = s_year.iloc[get_distances_vectorized(row['lat'], row['lon'], s_year['latitude'].values, s_year['longitude'].values) <= 50]['mag'].max() if not s_year.empty else 0
    return h_max if not pd.isna(h_max) else 0, s_max if not pd.isna(s_max) else 0

df[['max_wind_knots', 'max_seismic_mag']] = df.apply(lambda r: pd.Series(get_shocks(r)), axis=1)

In [3]:
df['target_pop_change'] = df.groupby('municipio')['total_population'].pct_change() * 100
df_clean = df.dropna(subset=['target_pop_change']).copy()

svi_cols = ['poverty_rate_pct', 'unemployment_rate_pct', 'no_hs_diploma_pct', 'disability_pct', 'no_vehicle_pct']
scaler = StandardScaler()
svi_scaled = scaler.fit_transform(df_clean[svi_cols])
pca = PCA(n_components=1)
df_clean['vulnerability_index'] = pca.fit_transform(svi_scaled)

features = ['vulnerability_index', 'median_income_real', 'over_65_pct', 'max_wind_knots', 'max_seismic_mag']
X = df_clean[features]
y = df_clean['target_pop_change']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/decomposition/_base.py:152: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/decomposition/_base.py:152: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/decomposition/_base.py:152: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T


In [4]:
# Train Random Forest
rf_model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

# Predict & Evaluate
y_pred = rf_model.predict(X_test)
print(f"Random Forest R-squared: {r2_score(y_test, y_pred):.4f}")
print(f"Random Forest RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}%")

Random Forest R-squared: -0.0249
Random Forest RMSE: 2.1531%


In [5]:
from sklearn.model_selection import GridSearchCV

# 1. Define the Parameter Grid
# We'll test different numbers of trees, tree depths, and splitting criteria
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'bootstrap': [True, False]
}

# 2. Initialize the Grid Search
# 'cv=5' means it will split the training data 5 times to validate each setting
grid_search = GridSearchCV(estimator=RandomForestRegressor(random_state=42), 
                           param_grid=param_grid, 
                           cv=5, 
                           n_jobs=-1, 
                           verbose=2, 
                           scoring='r2')

# 3. Execute the Search
print("Starting Hyperparameter Optimization...")
grid_search.fit(X_train, y_train)

# 4. Extract and Evaluate the Best Model
best_rf = grid_search.best_estimator_
y_opt_pred = best_rf.predict(X_test)

opt_r2 = r2_score(y_test, y_opt_pred)
opt_rmse = np.sqrt(mean_squared_error(y_test, y_opt_pred))

print("\n--- OPTIMIZED RANDOM FOREST PERFORMANCE ---")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Optimized R-squared: {opt_r2:.4f}")
print(f"Optimized RMSE:      {opt_rmse:.4f} %")
print("-" * 45)
print(f"Final Gain over Baseline (0.0144): {((opt_r2 - 0.0144) / 0.0144) * 100:.2f}%")

Starting Hyperparameter Optimization...
Fitting 5 folds for each of 72 candidates, totalling 360 fits
[CV] END bootstrap=True, max_depth=None, min_samples_split=2, n_estimators=100; total time=   0.2s
[CV] END bootstrap=True, max_depth=None, min_samples_split=2, n_estimators=100; total time=   0.2s
[CV] END bootstrap=True, max_depth=None, min_samples_split=2, n_estimators=100; total time=   0.2s
[CV] END bootstrap=True, max_depth=None, min_samples_split=2, n_estimators=100; total time=   0.2s
[CV] END bootstrap=True, max_depth=None, min_samples_split=2, n_estimators=100; total time=   0.2s
[CV] END bootstrap=True, max_depth=None, min_samples_split=2, n_estimators=200; total time=   0.4s
[CV] END bootstrap=True, max_depth=None, min_samples_split=2, n_estimators=200; total time=   0.4s
[CV] END bootstrap=True, max_depth=None, min_samples_split=2, n_estimators=200; total time=   0.4s
[CV] END bootstrap=True, max_depth=None, min_samples_split=2, n_estimators=200; total time=   0.4s
[CV] EN